### Workflow

Inspect quotient-splitting on fixed-weight decoder inputs, compare damping variants and run the compiled implementation. The selected trajectories help explain behavior; they do not establish an overall LER or runtime advantage.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/research/quotient'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/quotient_split_paths.json')


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations

DATA_DIR = _data_paths.run()
DATA_DIR.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD, CythonBPOSD, QuotientSplit
from src.decoders import Tesseract
from utils.quotient_split import *
from utils.trajectory_quotient_bp import build_quotient_model
from beliefmatching import detector_error_model_to_check_matrices
from utils.graph_vis import visualize_det_on_tanner_graph
from utils.graph_struct import build_tanner_graph_from_dem


In [ ]:
ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
WEIGHT_DIR = _data_paths.input('error_weight')
os.makedirs(WEIGHT_DIR, exist_ok=True)
num_errs = 33
num_dets = 14
num_obs = 2
max_iter = 60
code = ca_codes['CA_9_2_5']


## Hard Decision Trajectory

### Load weight-specific cases

Read the weight-2/weight-3 support tables and select a diagnostic disagreement. Keep the saved syndrome and true logical observables with the reconstructed decoding problem.


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2_data = pd.read_csv(_data_paths.input('weight_2_data_csv'), dtype=types)


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight3_data = pd.read_csv(_data_paths.input('weight_3_data_csv'), dtype=types)


In [ ]:
weight2_data


In [ ]:
mask = (weight2_data['pred_b'] != weight2_data['true']) & (weight2_data['pred_t'] == weight2_data['true'])
filtered_data = weight2_data[mask]
filtered_data


In [ ]:
filtered_data.shape


In [ ]:
row = filtered_data[(filtered_data['det'] == '11011000000110') & (filtered_data['err'] == '010000000100000000000000000000000')].iloc[-1]
error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()


In [ ]:
G, _ = build_tanner_graph_from_dem(dem)
visualize_det_on_tanner_graph(G, row['det'])


### Build and inspect the quotient model

Extract the parity-check/logical matrices and construct the quotient model and logical-neutral representative. These objects are used by the subsequent support-level trajectory comparisons.


In [ ]:
demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
H = demMatrices.check_matrix.toarray()
L = demMatrices.observables_matrix.toarray()
p = demMatrices.priors


In [ ]:
(H, L)


In [ ]:
(np.linalg.matrix_rank(H), np.linalg.matrix_rank(L), np.linalg.matrix_rank(np.vstack([H, L])))


In [ ]:
model = build_quotient_model(H, L)
print('H shape:', H.shape)
print('L shape:', L.shape)
print('rank(H):', model.rank_H)
print('dim ker(H):', model.K.shape[1])
print('degeneracy dimension:', model.D.shape[1])
print('logical rank:', model.logical_rank)
print('Q shape:', model.Q.shape)
print('LQ =')
print(L @ model.Q % 2)


In [ ]:
x_s = logical_neutral_representative(model.H, model.L, model.Q, str2bits(row['det']))
decoder = QuotientSplitDecoder(
    model.D,
    model.Q,
    x_s,
    p,
    list_width=2,
    internal_iterations=5,
    leaf_iterations=50,
    damping=0.0,
)
result = decoder.decode()

print(result.best_logical_label)
print(result.correction)
for leaf in result.leaves:
    print(leaf.logical_label, leaf.path_score, leaf.log_z_bethe)


In [ ]:
row['true']


### Run the initial support comparison

Evaluate the configured quotient-splitting trajectory and retain its failures for case inspection. The selected row below is an illustration from this support population.


In [ ]:
fails = 0
fail_cases = []
total = weight2_data.shape[0]

for _, row in weight2_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = demMatrices.check_matrix.toarray()
    L = demMatrices.observables_matrix.toarray()
    model = build_quotient_model(H, L)
    e = str2bits(row['err'])
    syndrome = str2bits(row['det'])
    true_observable = str2bits(row['true'])
    x_s = logical_neutral_representative(model.H, model.L, model.Q, str2bits(row['det']))
    decoder = QuotientSplitDecoder(
        model.D,
        model.Q,
        x_s,
        demMatrices.priors,
        list_width=2,
        internal_iterations=5,
        leaf_iterations=50,
        damping=0.0,
    )
    result = decoder.decode()

    if not np.array_equal(result.best_logical_label, true_observable):
        fails += 1
        fail_cases.append(row)

print(f'Failed: {fails}/{total}')


In [ ]:
fail_cases = pd.DataFrame(fail_cases)
fail_cases


In [ ]:
row = weight2_data.iloc[26]
error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
H = demMatrices.check_matrix.toarray()
L = demMatrices.observables_matrix.toarray()
p = demMatrices.priors
x_s = logical_neutral_representative(model.H, model.L, model.Q, str2bits(row['det']))
decoder = QuotientSplitDecoder(
    model.D,
    model.Q,
    x_s,
    p,
    list_width=2,
    internal_iterations=5,
    leaf_iterations=50,
    damping=0.2,
)
result = decoder.decode()

print(result.best_logical_label)
print(result.correction)
for leaf in result.leaves:
    print(leaf.logical_label, leaf.path_score, leaf.log_z_bethe)


with damping

### Repeat with damping

Run the damping variant and inspect the resulting failures before comparing individual examples. A changed failure count must be interpreted together with the tested settings and input cohort.


In [ ]:
fails = 0
fail_cases = []
total = weight2_data.shape[0]

for _, row in weight2_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = demMatrices.check_matrix.toarray()
    L = demMatrices.observables_matrix.toarray()
    model = build_quotient_model(H, L)
    e = str2bits(row['err'])
    syndrome = str2bits(row['det'])
    true_observable = str2bits(row['true'])
    x_s = logical_neutral_representative(model.H, model.L, model.Q, str2bits(row['det']))
    decoder = QuotientSplitDecoder(
        model.D,
        model.Q,
        x_s,
        demMatrices.priors,
        list_width=2,
        internal_iterations=5,
        leaf_iterations=50,
        damping=0.2,
    )
    result = decoder.decode()

    if not np.array_equal(result.best_logical_label, true_observable):
        fails += 1
        fail_cases.append(row)

print(f'Failed: {fails}/{total}')


In [ ]:
fail_cases = pd.DataFrame(fail_cases)
fail_cases


In [ ]:
row = weight3_data.iloc[31360]
error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
H = demMatrices.check_matrix.toarray()
L = demMatrices.observables_matrix.toarray()
p = demMatrices.priors
model = build_quotient_model(H, L)
x_s = logical_neutral_representative(model.H, model.L, model.Q, str2bits(row['det']))
decoder = QuotientSplitDecoder(
    model.D,
    model.Q,
    x_s,
    p,
    list_width=2,
    internal_iterations=10,
    leaf_iterations=60,
    damping=0.001,
)
result = decoder.decode()

print(result.best_logical_label)
print(result.correction, np.array_equal(H @ result.correction, str2bits(row['det'])))
for leaf in result.leaves:
    print(leaf.logical_label, leaf.path_score, leaf.log_z_bethe)


In [ ]:
weight3_data[(weight3_data['pred_b'] == weight3_data['true']) & (weight3_data['pred_t'] != weight3_data['true'])]


## CPP

### Compare the C++ implementation

Evaluate the existing compiled QuotientSplit decoder on the loaded fixed-weight tables. Later syndrome lookups illustrate ambiguous inputs and selected failures rather than a new unbiased benchmark.


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2_data = pd.read_csv(_data_paths.input('weight_2_data_csv'), dtype=types)


In [ ]:
fails = 0
total = weight2_data.shape[0]
for _, row in weight2_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    compiled_b = QuotientSplit(code, error_rate.p, damping=0.05).compile_decoder_for_dem(dem=dem)
    correction = compiled_b.decoder.decode_to_correction(str2bits(row['det']))
    correction_observable = compiled_b.decoder.L @ correction % 2
    true_observable = str2bits(row['true'])

    if not np.array_equal(correction_observable, true_observable):
        fails += 1

print(f'Failed: {fails}/{total}')


In [ ]:
weight2_data['det'].value_counts()


In [ ]:
weight2_data[weight2_data['det'] == '10100000000000']['true'].value_counts()


In [ ]:
types = {
    'p': float,
    'err': str,
    'det': str,
    'corr_b': str,
    'corr_t': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight3_data = pd.read_csv(_data_paths.input('weight_3_data_csv'), dtype=types)


In [ ]:
fails = 0
total = weight3_data.shape[0]
for _, row in weight3_data.iterrows():
    error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    compiled_b = QuotientSplit(code, error_rate.p).compile_decoder_for_dem(dem=dem)
    correction = compiled_b.decoder.decode_to_correction(str2bits(row['det']))
    correction_observable = compiled_b.decoder.L @ correction % 2
    true_observable = str2bits(row['true'])

    if not np.array_equal(correction_observable, true_observable):
        fails += 1

print(f'Failed: {fails}/{total}')


In [ ]:
weight3_data[weight3_data['pred_b'] != weight3_data['true']]


In [ ]:
weight3_data[weight3_data['det'] == '00011000000110']['true'].value_counts()


In [ ]:
weight3_data[weight3_data['det'] == '00000010110101']


### Inspect remaining weight-3 examples

Reconstruct and visualize the selected weight-3 rows after the aggregate support comparisons. Shared detector patterns can have multiple true logical explanations.


In [ ]:
row = weight3_data.iloc[5276]
error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()


In [ ]:
G, _ = build_tanner_graph_from_dem(dem)
visualize_det_on_tanner_graph(G, row['det'], other_events={('err', 'green', 'circle'): row['err']})


In [ ]:
row = weight3_data.iloc[5354]
G, _ = build_tanner_graph_from_dem(dem)
visualize_det_on_tanner_graph(G, row['det'], other_events={('err', 'green', 'circle'): row['err']})


In [ ]:
row = weight3_data.iloc[27100]
error_rate = ErrorRate(p=float(row['p']), p_meas=0.0)
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
compiled_b = QuotientSplit(code, error_rate.p).compile_decoder_for_dem(dem=dem)
correction = compiled_b.decoder.decode_to_correction(str2bits(row['det']))
correction_observable = compiled_b.decoder.L @ correction % 2
true_observable = str2bits(row['true'])

print('Correction observable:', correction_observable)
print('True observable:', true_observable)
